In [1]:
# import the library
from pynq import Overlay     # import the overlay
from pynq import allocate    # import for CMA (contingeous memory allocation)
from pynq import DefaultIP   # import the ip connector library for extension
from pynq import Interrupt
import asyncio
import numpy as np
import os
import subprocess
import re
import dfx4ml.magicSeq as magicSeq  # import the magic sequence library
import dfx4ml.dfxCtrl as dfxCtrl  # import the dfx control library
import dfx4ml.cap     as cap
import dfx4ml.memAlloc as dataAlloc  # import the memory allocation library
import dfx4ml.magicStreamDbg as mgsDb
import time

PRJ_NAME   = 'unetTest'  # the name of the bitstream
PRJ_DIR    = f'/home/xilinx/jupyter_notebooks/tanawin/{PRJ_NAME}/'
PRJ_HW_DIR = f'/home/xilinx/jupyter_notebooks/tanawin/{PRJ_NAME}/hw/'
PRJ_TC_DIR = f'/home/xilinx/jupyter_notebooks/tanawin/{PRJ_NAME}/tc/'

DFX_CONFIG_FILE = 'dfxCtrlMeta.txt'

MAGIC_STREAMER_IW = [14, 14]
MAGIC_STREAMER_WW = [32, 32]

FULL_BS_NAME    = 'system.bin'
PAR_BS_NAME_0   = 'skipper1.bin' ###### dma to magic stream 1
PAR_BS_NAME_1   = 'skipper2.bin' ###### magic stream 1 to magic stream 2
INPUT_DATA_NAME      = "inputX_1.npy"


AMT_QUERY       = 1
INPUT_SHAPE     = (AMT_QUERY, 4,4,1)  # 4*4* float32 = 64 bytes
# intermediate layer = (4*4*8* float32 = 512 bytes)
# intermediate layer = (4*4*8* float32 = 512 bytes)
OUTPUT_SHAPE    = (AMT_QUERY, 4,4,1)  # 4*4* float32 = 64 bytes
AMT_SLOT = 2

In [2]:
cap.changePLconfigMode("pcap", True)

CHANGE CMD STDOUT: 0xFFCA3008 0xFFFFFFFF 0x1

CHANGE CMD ERROR : 
--------------------------------
TRIGGER CMD STDOUT: 0xFFCA3008

TRIGGER CMD ERROR : 
--------------------------------
READ CMD STDOUT: 0x1

READ CMD ERROR : 
--------------------------------


In [3]:
#### load the overlay
overlay  = Overlay(PRJ_HW_DIR + FULL_BS_NAME)

In [4]:
#### create the interrupt pin
overlay.interrupt_pins

{'magicSeq/MagicSeqTop_0/hw_intr': {'controller': 'axi_intc_0',
  'index': 0,
  'fullpath': 'magicSeq/MagicSeqTop_0/hw_intr'},
 'axi_intc_0/intr': {'controller': 'axi_intc_0',
  'index': 0,
  'fullpath': 'axi_intc_0/intr'}}

In [5]:
my_interrupt = Interrupt('magicSeq/MagicSeqTop_0/hw_intr')  # index 0 from your mapping

In [6]:
#### get the device
dmaIp      = overlay.dataMovement.axi_dma_0
dfxCtrlIp  = overlay.PRcontroller.dfx_controller_0
magicSeqIp = overlay.magicSeq.MagicSeqTop_0
mgsDbgIps   = []

##### get magic streamer Debugger
dataMvHeir = overlay.dataMovement
for idx in range(len(MAGIC_STREAMER_IW)):
    streamGrp = getattr(dataMvHeir, f'streamGrp{idx}')
    mgsDebugIp = streamGrp.streamDbg_3
    mgsDbgIps.append(mgsDebugIp)

In [7]:
#### initialize magic streamer debuger
mgsDebugs = mgsDb.MagicStreamDbg(mgsDbgIps, MAGIC_STREAMER_IW, MAGIC_STREAMER_WW)
mgsDebugs.printDebugAll()

magic streamer debugger
-------- STREAM 0----------
state: STATUS_IDLE
amtLoad: 0 bytes/ 65536 bytes
amtStore: 0 bytes/ 65536 bytes
-------- STREAM 1----------
state: STATUS_IDLE
amtLoad: 0 bytes/ 65536 bytes
amtStore: 0 bytes/ 65536 bytes
----------------------------------


In [8]:
#### configure the dfx controller ip to match the address space
dfxCtrlIp.config(PRJ_HW_DIR + DFX_CONFIG_FILE)
print("regIdxSize = ", dfxCtrlIp.BLS_REGID)

regbank detect index (7, 8)
regbank detect index (2, 6)
regIdxSize =  5


In [9]:
### change reconfigure mode
cap.changePLconfigMode("icap", True)

CHANGE CMD STDOUT: 0xFFCA3008 0xFFFFFFFF 0x0

CHANGE CMD ERROR : 
--------------------------------
TRIGGER CMD STDOUT: 0xFFCA3008

TRIGGER CMD ERROR : 
--------------------------------
READ CMD STDOUT: 0x0

READ CMD ERROR : 
--------------------------------


In [10]:
dfxCtrlIp.printStatus()

[get status register] @ 0x0
>>status of the system vs0
-------> Is device shutdown:  1
-------> current error code:  0x0
-------> active RM_ID      :  0x0
-------> state      :  0x1


In [11]:
#### shutdown all system

magicSeqIp.shutdownEngine()
dfxCtrlIp .shutdownEngine()

[cmd] shutdown the engine
[cmd] shutdown successfully
shutdown dfx Controller
[set ctrl register] @ 0x0  with command  0x0


In [12]:
# get physical address of dma and dfx controller
dmaPhyAddr     =  overlay.ip_dict['dataMovement/axi_dma_0']['phys_addr']
dfxCtrlPhyAddr =  overlay.ip_dict['PRcontroller/dfx_controller_0']['phys_addr']

print("dma physical address: ", hex(dmaPhyAddr))
print("dfx  Ctrl physical address: ", hex(dfxCtrlPhyAddr))

dma physical address:  0xa0040000
dfx  Ctrl physical address:  0xa0020000


In [13]:
##### initialize magic seq
print("------ before init magic seq------")
print(magicSeqIp.printDebug())

print("------ init magic sequence METADATA bank 0 -------------------------")
magicSeqIp.setEndCnt(AMT_SLOT-1) ### use the last index
magicSeqIp.setDmaAddr(dmaPhyAddr)
magicSeqIp.setDfxAddr(dfxCtrlPhyAddr)
magicSeqIp.setIntrEna(1)
magicSeqIp.setIntr(1)  # woc  command 1 to set the interrupt to 0
magicSeqIp.setRoundTrip(0)  # set round trip to 0, no need to wait for the dma to finish
inputX = np.load(PRJ_TC_DIR + INPUT_DATA_NAME)
if(inputX.shape != INPUT_SHAPE):
    raise Exception(f"inputX shape is {inputX} expect {INPUT_SHAPE}")

#inputX = np.random.rand(*INPUT_SHAPE).astype(np.float32)
print("-------------init all data buffer -------------")
buf_input   , buf_input_phya   , buf_input_sz    = dataAlloc.allocDataUint(allocShape= INPUT_SHAPE, allocType= np.float32, inputX = inputX)
buf_out     , buf_out_phy      , buf_out_sz      = dataAlloc.allocDataUint(allocShape= OUTPUT_SHAPE  , allocType= np.float32)
buf_input.flush()
print("------------- init all bank 1 ------------------")
######                      srcPhyAddr    ,        srcSz,  dstPhyAddr,      dstSz,st,pr,loadMask, storeMask, intrMask 
magicSeqIp.setWholeSlot(0, [buf_input_phya, buf_input_sz,           0,          0, 0, 0,  0b0001,    0b0110, 0])
magicSeqIp.setWholeSlot(1, [             0,            0, buf_out_phy, buf_out_sz, 0, 0,  0b0110,    0b0001, 0])

print("------------- after init magic seq------")
print(magicSeqIp.printDebug())

------ before init magic seq------
----- MAIN STATUS ------------------
--------> STATUS =  SHUTDOWN
--------> MAINCNT =  0
--------> ENDCNT  =  0
--------> DMAADDR  =  0x0
--------> DFXADDR  =  0x0
--------> INTR_ENA =  0x0
--------> INTR     =  0x0
--------> ROUND_TRIP =  0x0
----- SLOT DATA ------------------
------> slot 0 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 1 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 2 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 3 :
      

In [14]:
##### initialize dfx controller
print("------ allocate bit steram CMA for each trigger ------")

######## set trigger 0
d0_ip_buf, d0_addr, d0_size = \
    dfxCtrlIp.allocateBitStreamCMA(PRJ_HW_DIR + PAR_BS_NAME_0)
######## set trigger 1
d1_ip_buf, d1_addr, d1_size = \
    dfxCtrlIp.allocateBitStreamCMA(PRJ_HW_DIR + PAR_BS_NAME_1)

------ allocate bit steram CMA for each trigger ------
>>allocateBitStream
opening file  /home/xilinx/jupyter_notebooks/tanawin/unetTest/hw/skipper1.bin
copying the data
copy complete
file size  18681736
---------------------------------
>>allocateBitStream
opening file  /home/xilinx/jupyter_notebooks/tanawin/unetTest/hw/skipper2.bin
copying the data
copy complete
file size  18726752
---------------------------------


In [15]:
##### initialize dfx controller2
dfxCtrlIp.setSimpleMetaData(0, d0_addr, d0_size)
dfxCtrlIp.setSimpleMetaData(1, d1_addr, d1_size)

setting RM Mapping to  0
[set RM MAP] @ 0x80  info  0x0
setting RM INFO to  0
control value for active low reset is  0x10
[get RM INFO] bsIdxAddr@ 0x100  ctrlAddr@ 0x104
setting BS INFO to  0  with streamAddress:  2017460224  with size:  18681736
[get BS INFO] streamAddr@ 0x184  sizeAddr@ 0x188
setting RM Mapping to  1
[set RM MAP] @ 0x84  info  0x1
setting RM INFO to  1
control value for active low reset is  0x10
[get RM INFO] bsIdxAddr@ 0x108  ctrlAddr@ 0x10c
setting BS INFO to  1  with streamAddress:  2036334592  with size:  18726752
[get BS INFO] streamAddr@ 0x194  sizeAddr@ 0x198


In [16]:
##### check dfx controller3
dfxCtrlIp.printStatus()
dfxCtrlIp.printSimpleMetaData(0)
dfxCtrlIp.printSimpleMetaData(1)

[get status register] @ 0x0
>>status of the system vs0
-------> Is device shutdown:  1
-------> current error code:  0x0
-------> active RM_ID      :  0x0
-------> state      :  0x1
get metadata info for row 0
[get RM MAP] @ 0x80
RM MAPPER:  0
[get RM INFO] bsIdxAddr@ 0x100  ctrlAddr@ 0x104
RM INFO  :  (0, 16)
[get BS INFO] streamAddr@ 0x184  sizeAddr@ 0x188
BS INFO  :  (0, 2017460224, 18681736)
get metadata info for row 1
[get RM MAP] @ 0x84
RM MAPPER:  1
[get RM INFO] bsIdxAddr@ 0x108  ctrlAddr@ 0x10c
RM INFO  :  (1, 16)
[get BS INFO] streamAddr@ 0x194  sizeAddr@ 0x198
BS INFO  :  (0, 2036334592, 18726752)


In [17]:
dfxCtrlIp.trigger(0)
dfxCtrlIp.restartNoStatus()

trig the rmId  0
[set Ctrl Trigger] @ 0x4
restart the dfx Controller with no status
[set ctrl register] @ 0x0  with command  0x1


In [18]:
dfxCtrlIp.printStatus()

[get status register] @ 0x0
>>status of the system vs0
-------> Is device shutdown:  0
-------> current error code:  0x0
-------> active RM_ID      :  0x0
-------> state      :  0x7


In [19]:
##### start dfx controller3
async def startExecAndWait4Intr():
    start_time = time.perf_counter()  # Start timing
    magicSeqIp.clearIntr()
    magicSeqIp.startEngine()
    while True:
        await my_interrupt.wait()
        end_time = time.perf_counter()
        print("interrupt")
        print(f"Elapsed time: {end_time - start_time:.6f} seconds")
        break

In [20]:
loop2 = asyncio.get_event_loop()

In [21]:
task2 = loop2.create_task(startExecAndWait4Intr())
loop2.run_until_complete(task2)

[cmd] clear the interrupt
[cmd] clear the interrupt successfully
[cmd] start the engine
[cmd] start the successfully
interrupt
Elapsed time: 0.094716 seconds


In [22]:
print(magicSeqIp.printDebug())

----- MAIN STATUS ------------------
--------> STATUS =  SHUTDOWN
--------> MAINCNT =  0
--------> ENDCNT  =  1
--------> DMAADDR  =  0xa0040000
--------> DFXADDR  =  0xa0020000
--------> INTR_ENA =  0x1
--------> INTR     =  0x1
--------> ROUND_TRIP =  0x1
----- SLOT DATA ------------------
------> slot 0 :
        srcAddr   : 0x18b9000,  srcSize   : 0x40
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x474444
        loadMask  : 0b1
        storeMask : 0b110
        stIntrMask: 0b110
------> slot 1 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x1879000,  desSize   : 0x40
        status    : 0x0
        profileCnt: 0x477038
        loadMask  : 0b110
        storeMask : 0b1
        stIntrMask: 0b1
------> slot 2 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 3

In [23]:
magicSeqIp.shutdownEngine()

[cmd] shutdown the engine
[cmd] shutdown successfully


In [24]:
print(magicSeqIp.printDebug())

----- MAIN STATUS ------------------
--------> STATUS =  SHUTDOWN
--------> MAINCNT =  0
--------> ENDCNT  =  1
--------> DMAADDR  =  0xa0040000
--------> DFXADDR  =  0xa0020000
--------> INTR_ENA =  0x1
--------> INTR     =  0x1
--------> ROUND_TRIP =  0x1
----- SLOT DATA ------------------
------> slot 0 :
        srcAddr   : 0x18b9000,  srcSize   : 0x40
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x474444
        loadMask  : 0b1
        storeMask : 0b110
        stIntrMask: 0b110
------> slot 1 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x1879000,  desSize   : 0x40
        status    : 0x0
        profileCnt: 0x477038
        loadMask  : 0b110
        storeMask : 0b1
        stIntrMask: 0b1
------> slot 2 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 3

In [25]:
mgsDebugs.printDebugAll()

magic streamer debugger
-------- STREAM 0----------
state: STATUS_IDLE
amtLoad: 64 bytes/ 65536 bytes
amtStore: 64 bytes/ 65536 bytes
-------- STREAM 1----------
state: STATUS_IDLE
amtLoad: 64 bytes/ 65536 bytes
amtStore: 64 bytes/ 65536 bytes
----------------------------------


In [26]:
buf_out.invalidate()
np_parRes = np.array(buf_out, dtype=np.float32)
print(np_parRes)

[[[[0.49609375]
   [0.5       ]
   [0.5       ]
   [0.5       ]]

  [[0.46875   ]
   [0.48828125]
   [0.4921875 ]
   [0.49609375]]

  [[0.4921875 ]
   [0.48828125]
   [0.4609375 ]
   [0.5       ]]

  [[0.4921875 ]
   [0.46875   ]
   [0.4609375 ]
   [0.48828125]]]]


In [27]:
print(buf_input)

[[[[0.7579385 ]
   [0.46447605]
   [0.4706111 ]
   [0.22119568]]

  [[0.36431968]
   [0.32350615]
   [0.06939422]
   [0.728553  ]]

  [[0.67173654]
   [0.78298676]
   [0.9412222 ]
   [0.34235054]]

  [[0.2812017 ]
   [0.5452858 ]
   [0.03334826]
   [0.7420243 ]]]]


In [28]:
print("dma write status: ", dmaIp.read(0x34))

dma write status:  4098


In [31]:
np.save(PRJ_DIR + "zcuOutput1.npy", np_parRes)